# Jump Search

Objetivo: Buscar em uma colecao **ordenada** dando saltos de tamanho fixo para localizar o bloco correto e, em seguida, aplicar busca linear dentro dele.

Complexidade:
- Tempo: O(sqrt(n)) com bloco otimo
- Espaco: O(1)

Categoria: Busca

## Diagrama do fluxo

```mermaid
flowchart TD
    A[Inicio: step = sqrt n] --> B[Saltar: prev=0, curr=step]
    B --> C{arr curr - 1 >= target ?}
    C -->|Nao, continuar saltando| D[prev=curr, curr=curr+step]
    D --> E{curr >= n ?}
    E -->|Sim| F[curr = n  ultimo bloco]
    E -->|Nao| C
    F --> G[Busca linear de prev ate curr]
    C -->|Sim, bloco encontrado| G
    G --> H{elemento == target ?}
    H -->|Sim| I[Retornar indice]
    H -->|Nao, avancar| J{chegou em curr ?}
    J -->|Nao| G
    J -->|Sim| K[Retornar -1]
```

## Fundamentos

Jump search e um algoritmo hibrido: combina saltos rapidos (como binary search) com busca linear (para ajuste fino). A ideia e pular blocos de tamanho `step` ate ultrapassar o alvo, depois varrer linearmente o bloco anterior.

O tamanho de bloco otimo e `sqrt(n)`. Com esse valor, o numero de saltos e `sqrt(n)` e o numero de comparacoes lineares no bloco e no maximo `sqrt(n) - 1`, resultando em O(sqrt(n)) total.

Jump search e especialmente util quando o **custo de voltar atras e alto** -- por exemplo, em listas ligadas (linked lists) ou em dispositivos de armazenamento sequencial onde retroceder e mais custoso que avancar. Nesses cenarios, binary search exigiria acesso a posicoes arbitrarias, enquanto jump search sempre avanca.

In [ ]:
import math


def jump_search(arr, target):
    """
    Searches for target in sorted array using jump search.

    Strategy:
      1. Jump ahead by sqrt(n) until the block that may contain target is found.
      2. Linear search backwards within that block.

    Args:
        arr: Sorted list of comparable elements.
        target: Value to search for.

    Returns:
        Index of target, or -1 if not found.

    Time:  O(sqrt(n))
    Space: O(1)
    """
    n = len(arr)
    if n == 0:
        return -1

    step = int(math.sqrt(n))  # optimal block size
    prev = 0

    # Phase 1: jump forward until arr[curr-1] >= target
    curr = step
    while curr < n and arr[curr - 1] < target:
        prev = curr
        curr += step

    # Phase 2: linear search within [prev, min(curr, n))
    end = min(curr, n)
    for i in range(prev, end):
        if arr[i] == target:
            return i
        if arr[i] > target:  # sorted: target cannot be further right
            break

    return -1


# Validation
sorted_data = list(range(0, 100, 3))  # [0, 3, 6, 9, ..., 99]
print(f"n={len(sorted_data)}, step=sqrt(n)={int(math.sqrt(len(sorted_data)))}")
print(f"Array (first 15): {sorted_data[:15]} ...")
print()
print(f"jump_search(arr, 33):  {jump_search(sorted_data, 33)}")   # 11
print(f"jump_search(arr, 0):   {jump_search(sorted_data, 0)}")    # 0
print(f"jump_search(arr, 99):  {jump_search(sorted_data, 99)}")   # 33
print(f"jump_search(arr, 1):   {jump_search(sorted_data, 1)}")    # -1 (not present)

## Visualizacao: padrao jump-then-linear

In [ ]:
def jump_search_verbose(arr, target):
    """
    Jump search with step-by-step trace output.
    Returns (index_or_minus1, jump_count, linear_count).
    """
    n = len(arr)
    if n == 0:
        return -1, 0, 0

    step = int(math.sqrt(n))
    prev = 0
    curr = step
    jump_count = 0
    linear_count = 0

    print(f"Searching for {target} in array of n={n}, step={step}")
    print("=" * 55)

    # Phase 1: jumps
    while curr < n and arr[curr - 1] < target:
        jump_count += 1
        print(f"  [JUMP {jump_count}] index {prev} -> {curr}: arr[{curr-1}]={arr[curr-1]} < {target}, keep jumping")
        prev = curr
        curr += step

    print(f"  [JUMP stop] candidate block: indices [{prev}, {min(curr, n)})")
    print()

    # Phase 2: linear
    end = min(curr, n)
    for i in range(prev, end):
        linear_count += 1
        if arr[i] == target:
            print(f"  [LINEAR {linear_count}] arr[{i}]={arr[i]} == {target}  FOUND")
            print(f"\nResult: index {i} | {jump_count} jumps + {linear_count} linear steps")
            return i, jump_count, linear_count
        print(f"  [LINEAR {linear_count}] arr[{i}]={arr[i]} != {target}")
        if arr[i] > target:
            break

    print(f"\nResult: -1 | {jump_count} jumps + {linear_count} linear steps")
    return -1, jump_count, linear_count


sample = list(range(1, 37, 2))  # [1, 3, 5, ..., 71]
jump_search_verbose(sample, 21)

## Por que o tamanho otimo do bloco e sqrt(n)

Seja `m` o tamanho do bloco. O custo total e:
- Saltos: `n/m` (numero de blocos percorridos)
- Busca linear: `m - 1` (no pior caso, varrer o bloco inteiro)
- Total: `n/m + m - 1`

Para minimizar essa expressao, derivamos em relacao a `m` e igualamos a zero:
- `-n/m² + 1 = 0` => `m = sqrt(n)`

Com `m = sqrt(n)`: custo = `sqrt(n) + sqrt(n) - 1 = 2*sqrt(n) - 1 = O(sqrt(n))`

In [ ]:
def jump_search_custom_step(arr, target, step):
    """
    Jump search with a user-defined block size.
    Returns (index_or_minus1, total_comparisons).
    """
    n = len(arr)
    if n == 0:
        return -1, 0

    comparisons = 0
    prev = 0
    curr = step

    while curr < n:
        comparisons += 1
        if arr[curr - 1] >= target:
            break
        prev = curr
        curr += step

    end = min(curr, n)
    for i in range(prev, end):
        comparisons += 1
        if arr[i] == target:
            return i, comparisons
        if arr[i] > target:
            break

    return -1, comparisons


# Compare different block sizes for n=400
n = 400
arr = list(range(n))
target = n - 1  # worst case: target at the end
optimal = int(math.sqrt(n))

print(f"n={n}, optimal step=sqrt({n})={optimal}, target={target} (worst case)")
print()
print(f"{'Block size':>12} | {'Comparisons':>13} | {'Note':>20}")
print("-" * 52)

for step in [1, 5, optimal, 40, 100, 200, 400]:
    _, comps = jump_search_custom_step(arr, target, step)
    note = "<-- optimal" if step == optimal else ""
    print(f"{step:>12} | {comps:>13} | {note:>20}")

## Benchmark: linear vs jump vs binary

In [ ]:
import timeit
import random


def linear_search_plain(arr, target):
    for i, v in enumerate(arr):
        if v == target:
            return i
    return -1


def binary_search_plain(arr, target):
    low, high = 0, len(arr) - 1
    while low <= high:
        mid = low + (high - low) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
    return -1


SIZES = [1_000, 10_000, 100_000]
RUNS = 300

print(f"{'n':>10} | {'linear (ms)':>12} | {'jump (ms)':>11} | {'binary (ms)':>12}")
print("-" * 55)

for n in SIZES:
    data = sorted(random.sample(range(n * 5), n))
    target = data[n // 2]  # mid element -- average case

    t_linear = timeit.timeit(
        lambda d=data, t=target: linear_search_plain(d, t), number=RUNS
    ) / RUNS * 1000

    t_jump = timeit.timeit(
        lambda d=data, t=target: jump_search(d, t), number=RUNS
    ) / RUNS * 1000

    t_binary = timeit.timeit(
        lambda d=data, t=target: binary_search_plain(d, t), number=RUNS
    ) / RUNS * 1000

    print(f"{n:>10,} | {t_linear:>12.4f} | {t_jump:>11.4f} | {t_binary:>12.4f}")

print()
print("Complexidades: linear=O(n), jump=O(sqrt n), binary=O(log n)")

## Comparacao com alternativas

| Algoritmo | Complexidade | Acesso aleatorio necessario | Retrocesso | Quando usar |
|---|---|---|---|---|
| Linear Search | O(n) | Nao | Nao | Dados nao ordenados |
| Jump Search | O(sqrt n) | Nao (apenas avanca) | Minimo (1 bloco) | Linked lists, acesso sequencial |
| Binary Search | O(log n) | Sim | Sim (pode ir qualquer posicao) | Arrays em memoria |

Jump search e superior ao linear e inferior ao binary em arrays. Sua vantagem sobre binary search emerge quando o custo de acessar uma posicao aleatoria e alto -- por exemplo em listas ligadas onde `arr[mid]` exige percorrer `mid` nos.

## Exercicios

**Desafio 1:** Implemente `jump_search_variable_blocks(arr, target, blocks)` que divide o array em `blocks` blocos de tamanho variavel (nao necessariamente sqrt(n)) e realiza a busca. Compare o numero de comparacoes para `blocks` em {2, 5, 10, sqrt(n), 20, 50} com n=1000 e target no final do array.

**Desafio 2:** Implemente jump search para uma **lista ligada simulada** (usando uma classe `Node` sem acesso por indice). Valide que o algoritmo funciona sem acesso aleatorio.

In [ ]:
# Desafio 1: blocos variaveis
def jump_search_variable_blocks(arr, target, blocks):
    """
    Jump search dividing arr into `blocks` equal-sized blocks.
    Returns (index_or_minus1, total_comparisons).
    """
    n = len(arr)
    if n == 0 or blocks <= 0:
        return -1, 0
    step = max(1, n // blocks)
    return jump_search_custom_step(arr, target, step)


n = 1000
arr = list(range(n))
target = n - 1  # worst case
optimal_blocks = int(math.sqrt(n))

print(f"n={n}, target={target} (worst case)")
print(f"{'blocks':>8} | {'step':>6} | {'comparisons':>13} | {'note':>15}")
print("-" * 50)

for blocks in [2, 5, 10, optimal_blocks, 20, 50, 100]:
    step = max(1, n // blocks)
    _, comps = jump_search_variable_blocks(arr, target, blocks)
    note = "<-- ~optimal" if blocks == optimal_blocks else ""
    print(f"{blocks:>8} | {step:>6} | {comps:>13} | {note:>15}")

print()

# Desafio 2: jump search em lista ligada simulada
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None


def build_linked_list(values):
    if not values:
        return None, 0
    head = Node(values[0])
    curr = head
    for v in values[1:]:
        curr.next = Node(v)
        curr = curr.next
    return head, len(values)


def jump_search_linked_list(head, n, target):
    """
    Jump search on a singly linked list (no random access).
    Always moves forward; linear backward scan within block.
    Returns node value and steps, or (-1, steps) if not found.
    """
    step = int(math.sqrt(n))
    prev = head
    curr = head
    count = 0

    # Phase 1: advance curr by step positions
    while curr is not None and curr.value < target:
        prev = curr
        # advance step nodes
        for _ in range(step):
            if curr.next is None:
                break
            curr = curr.next
            count += 1
        if curr.value < target and curr.next is not None:
            continue
        break

    # Phase 2: linear scan from prev to curr
    node = prev
    while node is not None:
        count += 1
        if node.value == target:
            return node.value, count
        if node.value > target:
            break
        if node is curr:
            break
        node = node.next

    return -1, count


values = list(range(0, 50, 2))  # [0, 2, 4, ..., 48]
head, length = build_linked_list(values)

for t in [20, 0, 48, 7]:
    result, steps = jump_search_linked_list(head, length, t)
    print(f"jump_search_linked_list(list, {t}): result={result}, steps={steps}")

## Referencias

1. Sartaj Sahni. "Data Structures, Algorithms, and Applications in Java". Silicon Press, 2000, pp. 231-235.
2. Cormen, T. H., et al. "Introduction to Algorithms" (3rd ed.). MIT Press, 2009, sec. 2.1.
3. Baeza-Yates, R., & Salinger, A. "Fast intersection algorithms for sorted sequences". In Algorithms and Applications, Springer, 2010.